In [ ]:
from urllib.robotparser import RobotFileParser
import json
import re
import requests
from bs4 import BeautifulSoup
import pdfplumber
from io import BytesIO
import unicodedata
import pandas as pd


In [2]:
def check_robots(url):
    """Check if scraping is allowed"""
    rp = RobotFileParser()
    rp.set_url('http://mtc-m16c.sid.inpe.br/robots.txt')
    rp.read()
    return rp.can_fetch('*', url)

if check_robots('http://mtc-m16c.sid.inpe.br/ibi/8JMKD3MGPDW34P/462CM9S'):
    print("✅ Scraping allowed")
else:
    print("❌ Scraping blocked by robots.txt")

✅ Scraping allowed


In [4]:
URL = "http://mtc-m16c.sid.inpe.br/ibi/8JMKD3MGPDW34P/462CM9S"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

In [5]:
page_articles_url = soup.find("frame", attrs={"name": "body"}).get("src")

In [6]:
response = requests.get(page_articles_url, headers=headers, timeout=30)
response.raise_for_status()

response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

In [7]:
def get_articles():
    urls = []
    for row in soup.find_all("table", "titleAuthorTABLE"):
        cell = row.find("font", class_="titleAuthorMiscFont")
        url = (cell.find("a")).get("href")
        urls.append(url) 
    return urls


In [8]:
articles = get_articles()
print(f"Quantidade de artigos: {len(articles)}")

Quantidade de artigos: 29


In [9]:
def get_metadatas():
    urls_metadata = []
    for article in articles:
        response = requests.get(article, headers=headers, timeout=30)
        response.raise_for_status()

        response.encoding = "utf-8"

        soup = BeautifulSoup(response.text, "html.parser")

        div = soup.find("div", class_="pageTitle")
        url = (div.find("a")).get("href")
        urls_metadata.append(url)

    return urls_metadata

In [10]:
url_metadatas = get_metadatas()
url_metadatas

['http://mtc-m16c.sid.inpe.br:80/col/sid.inpe.br/mtc-m18@80/2008/03.17.15.17.24/doc/mirrorget.cgi?languagebutton=en&metadatarepository=sid.inpe.br/mtc-m16c/2021/12.09.12.29.38&index=0&serveraddress=mtc-m16c.sid.inpe.br 804&choice=full&lastupdate=2023:01.30.13.08.16 sid.inpe.br/mtc-m18@80/2008/03.17.15.17 administrator {D 2021}&continue=yes&keywords=&accent=yes&case=yes&imageflag=0&mirrorgetflag=1',
 'http://mtc-m16c.sid.inpe.br:80/col/sid.inpe.br/mtc-m18@80/2008/03.17.15.17.24/doc/mirrorget.cgi?languagebutton=en&metadatarepository=sid.inpe.br/mtc-m16c/2021/12.09.12.27.16&index=0&serveraddress=mtc-m16c.sid.inpe.br 804&choice=full&lastupdate=2023:01.30.13.08.15 sid.inpe.br/mtc-m18@80/2008/03.17.15.17 administrator {D 2021}&continue=yes&keywords=&accent=yes&case=yes&imageflag=0&mirrorgetflag=1',
 'http://mtc-m16c.sid.inpe.br:80/col/sid.inpe.br/mtc-m18@80/2008/03.17.15.17.24/doc/mirrorget.cgi?languagebutton=en&metadatarepository=sid.inpe.br/mtc-m16c/2021/12.09.12.24.20&index=0&serveraddres

In [11]:
def get_field(name):
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) != 2:
            continue
        key = cells[0].get_text(" ", strip=True)
        if key == name:
            return cells[1].get_text(" ", strip=True)
    return None

In [12]:
def get_link(name):
    for row in soup.find_all("tr"):
        cells = row.find_all("td")

        if len(cells) != 2:
            continue

        key = cells[0].get_text(" ", strip=True)

        if key == name:
            a = cells[1].find("a")
            if a:
                return a["href"]

    return None

In [13]:
def get_pdf_link(doc_url):
    response = requests.get(doc_url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            return href

    return None

In [27]:
def get_pdfs():
    pdfs = {}

    for url_metadata in url_metadatas:

        response = requests.get(url_metadata, headers=headers, timeout=50)
        response.raise_for_status()
        response.encoding = response.apparent_encoding

        global soup
        soup = BeautifulSoup(response.text, "html.parser")

        titulo = (
            get_field("Title")
            .replace("æ", "'")
            .replace("Æ", "'")
            .replace("Ð", "ã")
            .replace("`", "'")
            .replace("Ó", "ç")
            .replace("„", "ã")
            .replace("Í", "ê")
            .replace("ę", "ê")
            .replace("ů", "ó")
            .replace("ă", "ã")
        )

        doc_url = get_link("doc Directory Content")

        pdf_url = get_pdf_link(doc_url)

        pdfs[titulo] = pdf_url

    return pdfs

In [28]:
urls_pdfs = get_pdfs()
urls_pdfs

{'Monitoring the spatiotemporal dynamics of surface water area of Goronyo Reservoir Sokoto, Nigeria using remote sensing': 'http://mtc-m16c.sid.inpe.br:80/attachment.cgi/sid.inpe.br/mtc-m16c/2021/12.09.12.29/doc/Abubakar_monitoring.pdf',
 'Towards an analytical and operational trajectory framework': 'http://mtc-m16c.sid.inpe.br:80/attachment.cgi/sid.inpe.br/mtc-m16c/2021/12.09.12.27/doc/Almeida_Towards.pdf',
 'Mapping irrigated rice using MSI/Sentinel-2 time series of vegetation indices and Random Forest': 'http://mtc-m16c.sid.inpe.br:80/attachment.cgi/sid.inpe.br/mtc-m16c/2021/12.09.12.24/doc/Araujo_Mapping.pdf',
 'Study on changing trends in climatic extremes in the Brazilian territory': 'http://mtc-m16c.sid.inpe.br:80/attachment.cgi/sid.inpe.br/mtc-m16c/2021/12.09.12.22/doc/Coelho_Study.pdf',
 'Effects of landscape fragmentation in the Protected Area of the Parque Estadual de Campos do Jordão - SP': 'http://mtc-m16c.sid.inpe.br:80/attachment.cgi/sid.inpe.br/mtc-m16c/2021/12.09.12.15

In [ ]:
def get_pages(pdf_url):
    """
    Retorna o número de páginas de um PDF.
    """

    try:
        r = requests.get(pdf_url, timeout=30)
        r.raise_for_status()

        with pdfplumber.open(BytesIO(r.content)) as pdf:
            return len(pdf.pages)

    except Exception as e:
        print(f"Erro ao ler PDF: {pdf_url}")
        print(e)
        return None

In [33]:
def fix_text(text):
    if text is None:
        return None

    return (
        text.replace("þ", "ç")
            .replace("~", "ã")
            .replace("`", "í")
            .replace("^", "ê")
            .replace("Ú", "é")
            .replace("Æ", '"')
            .replace("æ", '"')
    )

In [ ]:
def get_informations(url_metadata):
    title = get_field("Title")
    abstract = get_field("Abstract")
    article_type = get_field("Tertiary Type")
    language = get_field("Language")
    conference_name = get_field("Conference Name")
    author = get_field("Author")
    affiliations = get_field("Affiliation")
    editors = get_field("Editor")

    
    edition = None
    if conference_name:
        m = re.search(r",\s*(\d+)\s*\(", conference_name)
        if m:
            edition = int(m.group(1))

    metadata = {
        "title": title,
        "abstract": abstract,
        "article_type": article_type,
        "language": language,
        "event_edition": edition,
        "author": author,
        "affiliation": affiliations,
        "editors": editors
    }
    title = title.replace("`", "'")
    title = (
        title
        .replace("æ", "'")
        .replace("Æ", "'")
        .replace("Ð", "ã")
        .replace("Ó", "ç")
        .replace("æ", "'")
        .replace("Ń", "ã")
        .replace("ń", "ç")
        .replace("ă", "ã")
        .replace("„", "ã")
        .replace("ę", "ê")
        .replace("Í", "ê")
        .replace("ę", "ê")
        .replace("ů", "ó")
    )

    pdf_url = urls_pdfs[title]

    metadata["pages"] = get_pages(pdf_url)

    print(metadata)

    if metadata["article_type"] == "Full paper":
        metadata["article_type"] = "full"
    if metadata["article_type"] == "Short paper":
        metadata["article_type"] = "short"
    if metadata["language"] == "en":
        metadata["language"] = "english"
    if metadata["language"] == "pt":
        metadata["language"] = "portuguese"

    # Authors extraction and formatting
    raw_authors = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', metadata["author"])
    raw_authors = [fix_text(name.strip()) for name in raw_authors]
    # Convert "Last, First" to "First Last"
    formatted_authors = [
        f"{first_name.strip()} {last_name.strip()}" 
        if ',' in author 
        else author
        for author in raw_authors
        for last_name, first_name in [author.split(',', 1)] if ',' in author
    ]
    metadata["author"] = formatted_authors

  
    if metadata["affiliation"] != None:    
        raw_affiliations = re.findall(r'\d+\s+(.*?)(?=\s+\d+\s+|$)', metadata["affiliation"])
        raw_affiliations = [aff.strip() for aff in raw_affiliations]
        # Clean up encoding issues (replace \ufffd with –)
        formatted_affiliations = [
            aff.replace('\ufffd', '-').replace('�', '-') 
            for aff in raw_affiliations
        ]

        metadata["affiliation"] = formatted_affiliations

    else:
        metadata["affiliation"] = ["Universidade Federal de Pernambuco (UFPE)", "Ministério da Defesa Brasília", "Universidade Federal de Pernambuco (UFPE)", "Universidade Federal de Pernambuco (UFPE)"]

    # Editors extraction and formatting
    if metadata["editors"] is not None:

        raw_editors = re.findall(r'([^()]+?)\s*\([^()]+\)', metadata["editors"])

        formatted_editors = []

        for editor in raw_editors:
            editor = editor.strip()

            if "," in editor:
                last_name, first_name = editor.split(",", 1)
                editor = f"{first_name.strip()} {last_name.strip()}"

            formatted_editors.append(editor)

        metadata["editors"] = formatted_editors

    else:
        metadata["editors"] = []
    return metadata

In [ ]:
data = []
for url_metadata in url_metadatas:

    response = requests.get(url_metadata, headers=headers, timeout=30)
    response.raise_for_status()

    response.encoding = response.apparent_encoding

    soup = BeautifulSoup(response.text, "html.parser")

    informations = get_informations(url_metadata)
    print(json.dumps(informations, indent=4, ensure_ascii=False))
    data.append(informations)

with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

{'title': 'Monitoring the spatiotemporal dynamics of surface water area of Goronyo Reservoir Sokoto, Nigeria using remote sensing', 'abstract': 'Water stored in dams and reservoirs is essential element for hydrological cycle and other human activities like irrigation farming, fishing and transportation. Reservoirs in arid and semi-arid environments tend to change in volume and area extent over time as a result of natural and human factors causing water shortage. This study examines the spatiotemporal changes of Goronyo reservoir, Nigeria from 2000-2020. Landsat imageries were used to extract the surface water area using Modified Normalised Difference Water Index (MNDWI). The changes in the spatial and temporal pattern of the surface water over were obtained by calculating the differences in the surface area over the study period (2000-2020). The results show a continuous decrease in the surface water indicating loss of water. The surface area changed from 105.24km2 (98.35%) in 2000 to 

In [ ]:
with open("metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

arts = []

for article in metadata:
    
    arts.append({"titulo": article["title"],
           "abstract": article["abstract"],
           "n_paginas": article["pages"],
           "tipo_artigo": article["article_type"],
           "lingua": article["language"],
           "n_edicao": article["event_edition"]
    })

df = pd.DataFrame(arts)
df.insert(0, "id_artigo", range(1, len(df) + 1))

df.to_csv(
"artigos.csv",
columns=[
    "id_artigo",
    "titulo",
    "abstract",
    "n_paginas",
    "tipo_artigo",
    "lingua",
    "n_edicao"
],
index=False,
encoding="utf-8"
)

In [140]:
for article in metadata:
    if len(article["author"]) != len(article["affiliation"]):
        print(article["title"])
        print(len(article["author"]), len(article["affiliation"]))
        print(article["author"])
        print(article["affiliation"])

In [141]:
pessoas = {}
id_pessoa = 1

for article in metadata:
    # print(article["author"])
    for nome, instituicao in zip(article["author"], article["affiliation"]):
        if nome not in pessoas:
            pessoas[nome] = {
                "id_pessoa": id_pessoa,
                "nome": nome,
                "instituicao": instituicao
            }
            id_pessoa += 1

# inserir os editores
instituicoes = ["Instituto Nacional de Pesquisas Espaciais (INPE)", "Universidade Estadual do Rio de Janeiro (UERJ)"]
i = 0
for editor in metadata[1]["editors"]:
    if editor not in pessoas:
        pessoas[editor] = {
            "id_pessoa": id_pessoa,
            "nome": editor,
            "instituicao": instituicoes[i]
        }
        id_pessoa += 1
        i += 1

df = pd.DataFrame(pessoas.values())

df.to_csv(
    "pessoas.csv",
    index=False,
    encoding="utf-8"
)

In [142]:
pessoas

{'Bello Abubakar Abubakar': {'id_pessoa': 1,
  'nome': 'Bello Abubakar Abubakar',
  'instituicao': 'Nigerian Defence Academy'},
 'Sani Abubakar Abubakar': {'id_pessoa': 2,
  'nome': 'Sani Abubakar Abubakar',
  'instituicao': 'Kaduna Polytechnic'},
 'Damiăo Ribeiro de Almeida': {'id_pessoa': 3,
  'nome': 'Damiăo Ribeiro de Almeida',
  'instituicao': 'Universidade Federal de Campina Grande (UFCG)'},
 'Aillkeen Bezerra de Oliveira': {'id_pessoa': 4,
  'nome': 'Aillkeen Bezerra de Oliveira',
  'instituicao': 'Universidade Federal de Campina Grande (UFCG)'},
 'Samuel Pereira de Vasconcelos': {'id_pessoa': 5,
  'nome': 'Samuel Pereira de Vasconcelos',
  'instituicao': 'Universidade Federal de Campina Grande (UFCG)'},
 'Fábio Gomes de Andrade': {'id_pessoa': 6,
  'nome': 'Fábio Gomes de Andrade',
  'instituicao': 'Instituto Federal da Paraíba (IFPB)'},
 'Cláudio de Souza Baptista': {'id_pessoa': 7,
  'nome': 'Cláudio de Souza Baptista',
  'instituicao': 'Universidade Federal de Campina Grande

In [ ]:
autorias = []

id_autoria = 1

for id_artigo, article in enumerate(metadata, start=1):

    for ordem, nome in enumerate(article["author"], start=1):

        autorias.append({
            "id_autoria": id_autoria,
            "id_artigo": id_artigo,
            "id_pessoa": pessoas[nome]["id_pessoa"],
            "ordem_autoria": ordem,
            "autor_correspondente": False
        })

        id_autoria += 1

df = pd.DataFrame(autorias)

df.to_csv(
    "autoria.csv",
    index=False,
    encoding="utf-8"
)

In [153]:
organizadores = []

editores = metadata[1]["editors"]
n_edicao = metadata[1]["event_edition"]

funcoes = [
    "Program Chair",
    "General Chair"
]

id_organizador = 1

for i, nome in enumerate(editores):
    id_pessoa = pessoas.loc[
        pessoas["nome"] == nome,
        "id_pessoa"
    ].iloc[0]

    organizadores.append({
        "id_organizador": id_organizador,
        "id_pessoa": id_pessoa,
        "funcao": funcoes[i],
        "n_edicao": n_edicao
    })

    id_organizador += 1

df = pd.DataFrame(organizadores)

df.to_csv(
    "organizacao.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
def normalizar_nome(nome):
    nome = fix_text(nome)

    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join(c for c in nome if not unicodedata.combining(c))

    nome = nome.lower()
    nome = " ".join(nome.split())

    return nome

In [ ]:
pdf_url = "http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mtc-m16c/2021/12.17.00.48/doc/proceedings_geoinfo2021.pdf"
n_edicao = 22


def fix_text(text):
    if text is None:
        return ""

    return (
        text.replace("æ", "'")
        .replace("Æ", "'")
        .replace("Ð", "ã")
        .replace("Ó", "ç")
        .replace("æ", "'")
        .replace("Ń", "ã")
        .replace("ń", "ç")
        .replace("ă", "ã")
        .replace("„", "ã")
        .replace("ę", "ê")
        .replace("Í", "ê")
        .replace("ę", "ê")
        .replace("ů", "ó")
        .replace("~", "ã")
        .replace(",", "ç")
        .replace("^", "ê")
        .replace("¨", "ö")
        .replace("`a", "à")
        .replace("~a", "ã")
        .replace(",c", "ç")
    )


response = requests.get(pdf_url)
response.raise_for_status()

linhas = []

with pdfplumber.open(BytesIO(response.content)) as pdf:
    page = pdf.pages[4]

    largura = page.width
    altura = page.height

    esquerda = page.crop((0, 0, largura/2 - 10, altura))
    direita  = page.crop((largura/2 + 10, 0, largura, altura))

    texto = esquerda.extract_text() + "\n" + direita.extract_text()

    print(texto)


nomes = []

linhas = texto.splitlines()

for linha in linhas:
    linha = fix_text(linha.strip())

    if not linha:
        continue

    m = re.match(r"^(.*?)\s*\((.*?)\)$", linha)

    if m:
        nome = m.group(1).strip()
        instituicao = m.group(2).strip()
        nomes.append((nome, instituicao))

nomes = list(dict.fromkeys(nomes))

pessoas = pd.read_csv("pessoas.csv", encoding="utf-8")

ids = {
    normalizar_nome(nome): id_pessoa
    for nome, id_pessoa in zip(pessoas["nome"], pessoas["id_pessoa"])
}

proximo_id = pessoas["id_pessoa"].max() + 1

revisores = []

id_revisor = 1
alan_adicionado = False

for nome, instituicao in nomes:

    if "Alan" in nome:
        if alan_adicionado:
            continue

        revisores.append({
            "id_revisor": id_revisor,
            "id_pessoa": 114,
            "n_edicao": n_edicao
        })

        alan_adicionado = True
        id_revisor += 1
        continue

    chave = normalizar_nome(nome)

    if chave not in ids:

        ids[chave] = proximo_id

        pessoas.loc[len(pessoas)] = {
            "id_pessoa": proximo_id,
            "nome": fix_text(nome),
            "instituicao": fix_text(instituicao)
        }

        print(f"Pessoa adicionada: {nome}")

        proximo_id += 1

    revisores.append({
        "id_revisor": id_revisor,
        "id_pessoa": ids[chave],
        "n_edicao": n_edicao
    })

    id_revisor += 1

pessoas.to_csv(
    "pessoas.csv",
    index=False,
    encoding="utf-8"
)

pd.DataFrame(revisores).to_csv(
    "comite_cientifico.csv",
    index=False,
    encoding="utf-8"
)

print(f"{len(revisores)} revisores encontrados.")

Program Committee
Alan J. Salom˜ao Gra¸ca (UERJ)
Lubia Vinhas (INPE)
Alana Neves (INPE)
Alber S`anchez (INPE)
Alessandra Palmeiro (UFRJ)
Ana Clara Moura (UFMG)
Angelica di Maio (UFF
A. Miguel Monteiro (INPE)
Camilo Renn´o (INPE)
Carolina Pinho (UFABC)
Carla Macario (EMBRAPA)
Carlos Felgueiras (INPE)
Celso Silva Junior (INPE)
Claudia Krueger (UFPR)
Claudia Slutter (UFRS)
Claudio Barbosa (UFCG)
Claudio Campelo (UFCG)
Claudio Baptista (UFCG)
Clodoveu Davis-Jr (UFMG)
Cristina Ciferri (ICMC/USP)
Dalton Valeriano (INPE)
Daniel Vila (INPE)
Flavia Feitosa (UFABC)
Gabriela Miyoshi (UNESP)
Gilberto Queiroz (INPE)
Giovanni Comarela (UFES)
Haron Xaud (EMBRAPA)
Hugo Bendini (INPE)
Jorge Campos (UNIFACS)
Jose Quintanilha (USP)
Julio Esquerdo (EMBRAPA)
Julio Dalge (INPE)
Karine Ferreira (INPE)
Leonardo Santos (CEMADEN)
Leonardo Bins (INPE)
Liana Anderson (CEMADEN)
Lino Carvalho (UFRJ)
Lorena Santos (INPE)
Michel Chaves (INPE)
Michelle Picoli (U. C. Louvain, Belgiun)
M. Isabel Escada (INPE)
Manoel Sou